# 1.1. THỐNG KÊ MÔ TẢ
**Lab 05 – Phân tích khám phá dữ liệu (EDA)** · Trường Đại học Sài Gòn

Notebook gồm:
- **1.1.1** Ôn tập lý thuyết (trả lời các câu hỏi của đề)
- **1.1.3** Bài tập thực hành 1 – Red Wine Quality
- **1.1.4** Bài tập thực hành 2 – Pima Indians Diabetes

📦 **Dữ liệu được nạp trực tiếp từ Kaggle** bằng thư viện `kagglehub` (hàm `kaggle_read`), không cần tải file thủ công.
Lần chạy đầu cần có mạng; nếu Kaggle yêu cầu đăng nhập, thông báo lỗi sẽ hướng dẫn cách đăng nhập (`kagglehub.login()`).

In [ ]:
import os, sys, subprocess
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


# Dataset trên Kaggle (được nạp trực tiếp bằng kagglehub, không cần tải file thủ công)
KG_WINE = "uciml/red-wine-quality-cortez-et-al-2009"      # https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009
KG_DIAB = ["uciml/pima-indians-diabetes-database",        # https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
           "akshaydattatraykhare/diabetes-dataset",       # bản sao dự phòng trên Kaggle (cùng file diabetes.csv)
           "mathchi/diabetes-data-set"]                   # bản sao dự phòng trên Kaggle
# Nếu Kaggle không truy cập được (403, cần đăng nhập...), hàm kaggle_read sẽ dùng bản sao công khai trên GitHub:
URL_WINE_BACKUP = "https://raw.githubusercontent.com/plotly/datasets/master/winequality-red.csv"
URL_DIAB_BACKUP = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
KG_RETAIL = ["vijayuv/onlineretail", "carrie1/ecommerce-data"]   # Online Retail – thử lần lượt các dataset


def kaggle_read(slugs, filename, fallback_url=None, **kwargs):
    """Nạp dataset trực tiếp từ Kaggle thành DataFrame (kagglehub tự tải và lưu cache, không cần tải tay).
    slugs: 'chu-so-huu/ten-dataset' hoặc danh sách slug để thử lần lượt; filename: tên file CSV cần đọc.
    fallback_url: (tuỳ chọn) link CSV công khai, chỉ dùng khi Kaggle không truy cập được."""
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    if isinstance(slugs, str):
        slugs = [slugs]
    last_err = None
    for slug in slugs:
        try:
            folder = kagglehub.dataset_download(slug)
            files = [os.path.join(r, f) for r, _, fs in os.walk(folder) for f in fs if f.lower().endswith(".csv")]
            if not files:
                continue
            same = [f for f in files if os.path.basename(f).lower() == filename.lower()]
            path = same[0] if same else max(files, key=os.path.getsize)
            print(f"✅ Đã nạp dữ liệu từ Kaggle: {slug}  →  {os.path.basename(path)}")
            try:
                return pd.read_csv(path, encoding="utf-8-sig", **kwargs)
            except UnicodeDecodeError:
                return pd.read_csv(path, encoding="latin-1", **kwargs)
        except Exception as e:
            last_err = e
            print(f"   ⚠️ Không nạp được từ '{slug}': {repr(e)[:200]}")
    if fallback_url:
        print(f"↪️  Kaggle không truy cập được → dùng bản sao công khai: {fallback_url}")
        return pd.read_csv(fallback_url, encoding="utf-8-sig", **kwargs)
    raise RuntimeError(
        "Không nạp được dữ liệu từ Kaggle.\n"
        "  • Kiểm tra kết nối mạng.\n"
        "  • Nếu Kaggle yêu cầu đăng nhập: chạy `import kagglehub; kagglehub.login()` (cần API token: kaggle.com → Settings → API),\n"
        "    hoặc đặt biến môi trường KAGGLE_USERNAME và KAGGLE_KEY rồi chạy lại.\n"
        f"  Lỗi cuối: {last_err!r}"
    )


def describe_full(df):
    """Bảng thống kê mô tả đầy đủ cho các cột số."""
    num = df.select_dtypes(include="number")
    q1, q3 = num.quantile(0.25), num.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    res = pd.DataFrame({
        "count": num.count(),
        "missing": num.isna().sum(),
        "mean": num.mean(),
        "median": num.median(),
        "mode": num.mode().iloc[0],
        "var": num.var(),
        "std": num.std(),
        "min": num.min(),
        "max": num.max(),
        "range": num.max() - num.min(),
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "skew": num.skew(),
        "kurtosis": num.kurt(),
        "n_outliers(IQR)": ((num < low) | (num > high)).sum(),
    })
    return res

---
# 1.1.1. Ôn tập lý thuyết

**1. Thống kê mô tả là gì? Khác gì thống kê suy luận?**
Thống kê mô tả *tóm tắt và mô tả* dữ liệu đang có (bảng, chỉ số, biểu đồ) mà không kết luận gì vượt ra ngoài mẫu. Thống kê suy luận dùng mẫu để *suy ra* tính chất của tổng thể (ước lượng, kiểm định giả thuyết, khoảng tin cậy, p-value).

**2. Các thước đo chính dùng để làm gì? Khi nào dùng trung vị thay vì trung bình?**
- *Xu hướng trung tâm*: mean (trung bình), median (trung vị), mode (yếu vị – giá trị xuất hiện nhiều nhất).
- *Độ phân tán*: phương sai, độ lệch chuẩn, range, IQR.
- Dùng **median** khi phân bố lệch mạnh hoặc có ngoại lai (ví dụ giá nhà, thu nhập), vì mean bị kéo về phía đuôi dài còn median thì bền vững.

**3. Xác định phân bố của dữ liệu; các loại phân bố phổ biến?**
Vẽ histogram/KDE/boxplot, so sánh mean với median, tính skewness và kurtosis, dùng Q-Q plot hoặc kiểm định chuẩn (Shapiro–Wilk…).
- *Chuẩn (đối xứng)*: mean ≈ median, skew ≈ 0.
- *Lệch phải (positive skew)*: đuôi dài bên phải, mean > median.
- *Lệch trái (negative skew)*: đuôi dài bên trái, mean < median.
- Ngoài ra: đều, hai đỉnh (bimodal), mũ…

**4. Độ lệch chuẩn và range đánh giá độ phân tán thế nào?**
Độ lệch chuẩn đo mức dao động *trung bình* của dữ liệu quanh mean (cùng đơn vị với dữ liệu), dùng được mọi giá trị nên ổn định hơn. Range = max − min chỉ phụ thuộc 2 giá trị biên nên rất nhạy với ngoại lai.

**5. Q1, Q2, Q3 trong boxplot?**
Q1 (25%): 1/4 dữ liệu nhỏ hơn giá trị này. Q2 = median (50%). Q3 (75%). Hộp trải từ Q1 đến Q3 (IQR = Q3 − Q1); "râu" kéo đến các giá trị trong khoảng [Q1 − 1.5·IQR, Q3 + 1.5·IQR]; điểm nằm ngoài là ngoại lai.

**6. Xử lý giá trị thiếu trước khi tính thống kê?**
Kiểm tra `isnull().sum()`, tìm hiểu *vì sao* thiếu. Có thể: xoá dòng/cột (`dropna`), điền mean/median/mode, điền theo nhóm, nội suy (dữ liệu chuỗi thời gian), hoặc đánh dấu thiếu thành một cột riêng. Lưu ý các hàm numpy như `np.median` trả về `nan` nếu còn NaN, còn pandas mặc định bỏ qua NaN.

**7. Đọc histogram / boxplot từ dữ liệu thực tế?**
- *Histogram*: nhìn hình dạng (đối xứng/lệch), số đỉnh, khoảng giá trị tập trung, khoảng trống, ngoại lai.
- *Boxplot*: vạch giữa hộp là median, độ rộng hộp là IQR, râu là độ trải, chấm lẻ là ngoại lai; median lệch về một phía của hộp cho thấy phân bố lệch.

**8. Gặp ngoại lai thì xử lý thế nào?**
Trước hết *xác định* (IQR, z-score, boxplot) rồi *tìm nguyên nhân* (lỗi nhập liệu hay giá trị thật hiếm gặp). Nếu là lỗi → sửa/loại. Nếu là giá trị thật → giữ lại và dùng thước đo bền vững (median, IQR), hoặc winsorize/biến đổi log. Không nên xoá chỉ vì "trông lạ".

---
# 1.1.3. Bài tập thực hành 1 – Thống kê mô tả dữ liệu Red Wine Quality

📦 **Dữ liệu (Kaggle):** https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009
(notebook tham khảo: https://www.kaggle.com/code/eisgandar/red-wine-quality-eda-classification)
→ nạp trực tiếp bằng `kaggle_read` ở cell dưới.

In [ ]:
wine = kaggle_read(KG_WINE, "winequality-red.csv", fallback_url=URL_WINE_BACKUP, sep=None, engine="python")

print("Kích thước:", wine.shape)
display(wine.head())
wine.info()

In [ ]:
# Chất lượng dữ liệu
print("Giá trị thiếu:\n", wine.isnull().sum().sum(), "ô thiếu")
print("Số dòng trùng lặp:", wine.duplicated().sum())

# Thống kê mô tả cơ bản của pandas
display(wine.describe())

In [ ]:
# Thống kê mô tả đầy đủ: mean, median, mode, var, std, range, Q1, Q3, IQR, skew, kurtosis, số ngoại lai
wine_stats = describe_full(wine)
display(wine_stats)

In [ ]:
# Biến mục tiêu: quality
print("Phân bố quality (số lượng):")
display(wine["quality"].value_counts().sort_index().to_frame("count").assign(
    percent=lambda d: (d["count"] / d["count"].sum() * 100).round(2)))

# Giá trị trung bình của từng đặc trưng theo mức quality
display(wine.groupby("quality").mean().round(3))

# Tương quan của các đặc trưng với quality
display(wine.corr()["quality"].drop("quality").sort_values(ascending=False).to_frame("corr với quality"))

In [ ]:
# Cột nào lệch mạnh nhất? (|skew| > 1 thường coi là lệch rõ rệt)
skewed = wine_stats["skew"].abs().sort_values(ascending=False)
display(skewed.to_frame("|skew|").head(5))

**Gợi ý nhận xét (tự điền theo kết quả chạy):**
- Dữ liệu có thiếu / trùng lặp không? Xử lý thế nào?
- Các đặc trưng nào lệch phải (`skew` lớn, mean > median) và có nhiều ngoại lai (`residual sugar`, `chlorides`, `sulphates`…)?
- `quality` tập trung ở những mức nào? Dữ liệu có mất cân bằng không?
- Đặc trưng nào tương quan mạnh nhất với `quality` (thường là `alcohol`, `volatile acidity`)?

---
# 1.1.4. Bài tập thực hành 2 – Thống kê mô tả dữ liệu bệnh tiểu đường (Pima Indians)

📦 **Dữ liệu (Kaggle):** https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
(notebook tham khảo: https://www.kaggle.com/code/vincentlugat/pima-indians-diabetes-eda-prediction-0-906)
→ nạp trực tiếp bằng `kaggle_read` ở cell dưới.

In [ ]:
diab = kaggle_read(KG_DIAB, "diabetes.csv", fallback_url=URL_DIAB_BACKUP)
print("Kích thước:", diab.shape)
display(diab.head())
diab.info()
print("Thiếu (NaN):", diab.isnull().sum().sum(), "| Trùng lặp:", diab.duplicated().sum())

In [ ]:
display(describe_full(diab))

> ⚠️ Ở bộ dữ liệu này, các cột `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI` **không thể bằng 0 về mặt sinh học**.
> Số `0` thực chất là *giá trị thiếu được mã hoá thành 0* nên `isnull()` không phát hiện ra. Ta đổi `0 → NaN` rồi mới thống kê.

In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Số giá trị 0 ở mỗi cột:")
display((diab[zero_cols] == 0).sum().to_frame("số giá trị 0"))

diab_clean = diab.copy()
diab_clean[zero_cols] = diab_clean[zero_cols].replace(0, np.nan)
print("Số NaN sau khi đổi 0 → NaN:")
display(diab_clean.isna().sum().to_frame("missing"))

# So sánh mean/median trước và sau khi xử lý
cmp = pd.DataFrame({
    "mean (gốc)": diab[zero_cols].mean(),
    "mean (bỏ 0)": diab_clean[zero_cols].mean(),
    "median (gốc)": diab[zero_cols].median(),
    "median (bỏ 0)": diab_clean[zero_cols].median(),
})
display(cmp)

In [ ]:
# Điền thiếu bằng median (có thể điền theo nhóm Outcome để chính xác hơn)
diab_filled = diab_clean.copy()
for col in zero_cols:
    diab_filled[col] = diab_filled[col].fillna(diab_filled.groupby("Outcome")[col].transform("median"))

print("Sau khi điền, còn thiếu:", diab_filled.isna().sum().sum())
display(describe_full(diab_filled))

In [ ]:
# Biến mục tiêu Outcome (0: không bệnh, 1: có bệnh) và đặc trưng theo nhóm
display(diab_filled["Outcome"].value_counts().to_frame("count").assign(
    percent=lambda d: (d["count"] / d["count"].sum() * 100).round(2)))

display(diab_filled.groupby("Outcome").agg(["mean", "median"]).T)

display(diab_filled.corr()["Outcome"].drop("Outcome").sort_values(ascending=False).to_frame("corr với Outcome"))

**Gợi ý nhận xét:** tỉ lệ người bệnh (`Outcome = 1`) khoảng 1/3, `Glucose` và `BMI` thường tương quan mạnh nhất với `Outcome`;
`Insulin` và `DiabetesPedigreeFunction` lệch phải mạnh, nhiều ngoại lai; nhóm bệnh có `Glucose` trung bình cao hơn rõ rệt.